In [2]:
import pandas as pd
import multiprocessing as mp
from tqdm import tqdm
from protocols import utils, functions

%load_ext autoreload
%autoreload 2

In [6]:
# the path to the CRyPTIC v3.1.0 data tables (including the DST_MEASUREMENTS_+.pkl which has DST for the validation samples appended)
cryptic_tables_path = '/Users/fowler/Dropbox/files/cryptic/cryptic-release-three/cryptic-tables-v3.1.0/'

# whether to run the processes which take a long time
run_long_processes = False

# number of cores
n_cores = 6

drug_genes = {
    'BDQ': {"genes": ["Rv0678", "atpE", "pepQ"], "phylogenetic": []},
    'CFZ': {"genes": ["Rv0678", "atpE", "pepQ"], "phylogenetic": []},
    "AMI": {"genes": ["eis", "rrs"], "phylogenetic": []},
    "CAP": {"genes": ["rrs", "tlyA"], "phylogenetic": []},
    "DLM": {"genes": ["ddn"], "phylogenetic": []},
    "EMB": {"genes": ["embA", "embB"], "phylogenetic": []},
    "ETH": {"genes": ["ethA", "fabG1", "inhA"], "phylogenetic": []},
    "INH": {"genes": ["katG", "inhA", "ahpC", "fabG1"], "phylogenetic": []},
    "KAN": {"genes": ["eis", "rrs"], "phylogenetic": []},
    "LEV": {"genes": ["gyrA", "gyrB"], "phylogenetic": ['gyrA@S95T']},
    "LZD": {"genes": ["rplC"], "phylogenetic": []},
    "MXF": {"genes": ["gyrA", "gyrB"], "phylogenetic": ['gyrA@S95T']},
    "RIF": {"genes": ["rpoB"], "phylogenetic": []},
    "STM": {"genes": ["gid", "rpsL", "rrs"], "phylogenetic": []},
    'PZA': {"genes": ["pncA"], "phylogenetic": []},    
}

who_drugs = list(pd.read_csv('data/who2_drugs.csv').drug)

In [7]:
def process_performance(args):
    drug, genes, catalogues, dataset, training, FRS = args
    results = []

    if drug in who_drugs:
        
        mutations = functions.prep_mutations(
            'data/mutations-v3.1.0/', 
            genes, 
            version='v3.1.0', 
            mut_path=cryptic_tables_path+'MUTATIONS.parquet', 
            var_path=cryptic_tables_path+'VARIANTS.parquet',
            train=training
        )
        if dataset == 'training':
            version = 'v1.0'
            validation = False
        elif dataset == 'all':
            version = 'v3.0'
            validation = False
        elif dataset == 'validation':    
            version = 'v3.0'
            validation = True
        else:
            raise ValueError('dataset must be one of "training", "all", or "validation"')

        phenotypes = functions.prep_phenotypes(
            drug,
            cryptic_tables_path+'DST_MEASUREMENTS_+.pkl',
            cryptic_tables_path+'GENOMES.parquet',
            cryptic_tables_path+'WGS_SAMPLES.parquet',
            version,
            validation=validation
        )
        phenotypes.set_index('UNIQUEID', inplace=True)
        mutations.set_index('UNIQUEID', inplace=True)
        all_data = phenotypes.join(mutations[mutations.FRS >= FRS], how='left')
        all_data.reset_index(inplace=True)

        if len(all_data)>0:

            for cat_name in catalogues:

                _, cov, sens, spec, sens2, spec2 = utils.piezo_predict(iso_df=all_data, drug=drug, catalogue_file=catalogues[cat_name])

                results.append({
                    'DRUG': drug,
                    'catalogue': cat_name,
                    'SENSITIVITY': sens,
                    'SPECIFICITY': spec,
                    'COVERAGE': cov,
                    'SENSITIVITY2': sens2,
                    'SPECIFICITY2': spec2,
                })

    return results

def parallel_performance_evaluation(drug_genes, catalogues, dataset, training, frs):
    tasks = [(drug, data['genes'], catalogues, dataset, training, frs) for drug, data in drug_genes.items()]
    
    # Works on Mac/Linux
    ctx = mp.get_context("fork") 
    
    # Don't use more workers than tasks
    num_workers = min(n_cores, len(tasks))  

    # Run in parallel
    with ctx.Pool(num_workers) as pool:
        all_results = list(tqdm(pool.imap(process_performance, tasks), total=len(tasks)))
    
    results_df = pd.DataFrame([item for sublist in all_results for item in sublist])

    return results_df



In [8]:
if run_long_processes:

    catalogues = {
        'WHOv1': 'catalogues/whov1/NC_000962.3_WHO-UCN-GTB-PCI-2021.7_v1.2_GARC1_RUS.csv',
        'WHOv2': 'catalogues/whov2/NC_000962.3_WHO-UCN-TB-2023.5_v2.0_GARC1_RFUS.csv',
        'catomatic_v1': "catalogues/catomatic_v1.csv",
    }

    results_all = parallel_performance_evaluation(drug_genes, catalogues, dataset='training', training=False, frs=0.1)
    results_all.to_csv('results/performance/whov1_whov2_cat1_training.csv')

In [9]:
if run_long_processes:

    catalogues = {
        'WHOv1': 'catalogues/whov1/NC_000962.3_WHO-UCN-GTB-PCI-2021.7_v1.2_GARC1_RUS.csv',
        'WHOv2': 'catalogues/whov2/NC_000962.3_WHO-UCN-TB-2023.5_v2.0_GARC1_RFUS.csv',
        'catomatic_v1': "catalogues/catomatic_v1.csv",
    }

    results_all = parallel_performance_evaluation(drug_genes, catalogues, dataset='validation', training=False, frs=0.1)
    results_all.to_csv('results/performance/whov1_whov2_cat1_validation.csv')

In [10]:
phenotypes = functions.prep_phenotypes(
    'RIF',
    cryptic_tables_path+'DST_MEASUREMENTS_+.pkl',
    cryptic_tables_path+'GENOMES.parquet',
    cryptic_tables_path+'WGS_SAMPLES.parquet',
    'v3.0',
    validation=True
)
phenotypes.set_index('UNIQUEID', inplace=True)
phenotypes

,DRUG,PHENOTYPE,METHOD_MIC,METHOD_3
UNIQUEID,,,,
site.ENA.subj.ERR036186.lab.1.iso.1,RIF,S,NaN,NaN
site.ENA.subj.ERR036187.lab.1.iso.1,RIF,S,NaN,NaN
site.ENA.subj.ERR036188.lab.1.iso.1,RIF,S,NaN,NaN
site.ENA.subj.ERR036189.lab.1.iso.1,RIF,S,NaN,NaN
site.ENA.subj.ERR036190.lab.1.iso.1,RIF,S,NaN,NaN
...,...,...,...,...
site.ENA.subj.SRR8651589.lab.1.iso.1,RIF,R,NaN,NaN
site.ENA.subj.SRR8651594.lab.1.iso.1,RIF,R,NaN,NaN
site.ENA.subj.SRR8651616.lab.1.iso.1,RIF,R,NaN,NaN


In [11]:
mutations = functions.prep_mutations(
    'data/mutations-v3.1.0/', 
    ['rpoB'], 
    version='v3.1.0', 
    mut_path=cryptic_tables_path+'MUTATIONS.parquet', 
    var_path=cryptic_tables_path+'VARIANTS.parquet',
    train=False
)
mutations.set_index('UNIQUEID', inplace=True)
mutations

,MUTATION,FRS
UNIQUEID,,
site.02.subj.0069.lab.22A019.iso.1,rpoB@S450L,1.000000
site.02.subj.0069.lab.22A019.iso.1,rpoB@A1075A,1.000000
site.ENA.subj.SAMEA2533644.lab.1.iso.1,rpoB@A1075A,1.000000
site.07.subj.B7463DB0-52E0-4276-9C40-C5EF6F799A6C.lab.B7463DB0-52E0-4276-9C40-C5EF6F799A6C.iso.1,rpoB@A1075A,1.000000
site.03.subj.6236-05_LIB12062.lab.6236-05_LIB12062.iso.1,rpoB@S450L,1.000000
...,...,...
site.ENA.subj.SRR6824487.lab.1.iso.1,rpoB@C701W,0.665236
site.ENA.subj.SRR5551664.lab.1.iso.1,rpoB@S450L,1.000000
site.ENA.subj.SRR5551664.lab.1.iso.1,rpoB@A1075A,1.000000


In [12]:
all_data = phenotypes.join(mutations[mutations.FRS >= 0.1], how='left')
all_data.reset_index(inplace=True)
all_data

,UNIQUEID,DRUG,PHENOTYPE,METHOD_MIC,METHOD_3,MUTATION,FRS
0,site.ENA.subj.ERR036186.lab.1.iso.1,RIF,S,NaN,NaN,rpoB@A1075A,1.000000
1,site.ENA.subj.ERR036186.lab.1.iso.1,RIF,S,NaN,NaN,rpoB@G1010G,0.272727
2,site.ENA.subj.ERR036187.lab.1.iso.1,RIF,S,NaN,NaN,rpoB@G1010G,0.333333
3,site.ENA.subj.ERR036188.lab.1.iso.1,RIF,S,NaN,NaN,rpoB@S450L,1.000000
4,site.ENA.subj.ERR036188.lab.1.iso.1,RIF,S,NaN,NaN,rpoB@V970M,1.000000
...,...,...,...,...,...,...,...
7823,site.ENA.subj.SRR8651654.lab.1.iso.1,RIF,R,NaN,NaN,rpoB@T400T,1.000000
7824,site.ENA.subj.SRR8651654.lab.1.iso.1,RIF,R,NaN,NaN,rpoB@D435V,1.000000
7825,site.ENA.subj.SRR8651654.lab.1.iso.1,RIF,R,NaN,NaN,rpoB@A1075A,1.000000
7826,site.ENA.subj.SRR8651662.lab.1.iso.1,RIF,R,NaN,NaN,rpoB@S450L,1.000000


In [13]:
ids = all_data["UNIQUEID"].unique().tolist()
labels = all_data.groupby("UNIQUEID")["PHENOTYPE"].first().reindex(ids).tolist()
len(labels)

4628

In [29]:
catalogues = {
        'WHOv1': 'catalogues/whov1/NC_000962.3_WHO-UCN-GTB-PCI-2021.7_v1.2_GARC1_RUS.csv',
        'WHOv2': 'catalogues/whov2/NC_000962.3_WHO-UCN-TB-2023.5_v2.0_GARC1_RFUS.csv',
        'CATv1': "catalogues/catomatic_v1.csv",
    }

In [30]:
results = []
for cat in ['WHOv1', 'WHOv2', 'CATv1']:
    for drug in ['RIF', 'INH']:
        (i, a, b) = utils.piezo_predict(iso_df=all_data, drug=drug, catalogue_file=catalogues[cat], return_predictions=True)
        df2 = pd.DataFrame.from_dict({'id':i, 'prediction':b, 'label':a})
        df2['DRUG'] = drug
        df2['CATALOGUE'] = cat
        results.append(df2)
df = pd.concat(results)
df

,id,prediction,label,DRUG,CATALOGUE
0,site.ENA.subj.ERR036186.lab.1.iso.1,S,S,RIF,WHOv1
1,site.ENA.subj.ERR036187.lab.1.iso.1,S,S,RIF,WHOv1
2,site.ENA.subj.ERR036188.lab.1.iso.1,R,S,RIF,WHOv1
3,site.ENA.subj.ERR036189.lab.1.iso.1,S,S,RIF,WHOv1
4,site.ENA.subj.ERR036190.lab.1.iso.1,U,S,RIF,WHOv1
...,...,...,...,...,...
4623,site.ENA.subj.SRR8651589.lab.1.iso.1,S,R,INH,CATv1
4624,site.ENA.subj.SRR8651594.lab.1.iso.1,S,R,INH,CATv1
4625,site.ENA.subj.SRR8651616.lab.1.iso.1,S,R,INH,CATv1
4626,site.ENA.subj.SRR8651654.lab.1.iso.1,S,R,INH,CATv1


In [31]:
df.to_csv('poster.csv')

In [33]:
df[:3]

,id,prediction,label,DRUG,CATALOGUE
0,site.ENA.subj.ERR036186.lab.1.iso.1,S,S,RIF,WHOv1
1,site.ENA.subj.ERR036187.lab.1.iso.1,S,S,RIF,WHOv1
2,site.ENA.subj.ERR036188.lab.1.iso.1,R,S,RIF,WHOv1


In [62]:
foo = df.pivot(index=['id','CATALOGUE'], columns='DRUG', values=['prediction','label'])
foo

prediction     label    
DRUG                                                  INH RIF   INH RIF
id                                   CATALOGUE                         
site.ENA.subj.ERR036186.lab.1.iso.1  CATv1              S   S     S   S
                                     WHOv1              S   S     S   S
                                     WHOv2              S   S     S   S
site.ENA.subj.ERR036187.lab.1.iso.1  CATv1              S   S     S   S
                                     WHOv1              S   S     S   S
...                                                   ...  ..   ...  ..
site.ENA.subj.SRR8651654.lab.1.iso.1 WHOv1              S   R     R   R
                                     WHOv2              S   R     R   R
site.ENA.subj.SRR8651662.lab.1.iso.1 CATv1              S   R     R   R
                                     WHOv1              S   R     R   R
                                     WHOv2              S   R     R   R

[13884 rows x 4 columns]

In [66]:
foo[('prediction', 'INH')].value_counts()

(prediction, INH)
S    13884
Name: count, dtype: int64

In [54]:
def process(row):
    mdr = False
    agree = False
    if row.label.INH == 'R' and row.label.RIF == 'R':
        mdr = True
        if row.prediction.INH == 'R' and row.prediction.RIF == 'R':
            agree = True 

    return pd.Series([mdr,agree])

foo[['MDR', 'AGREE']] = foo.apply(process, axis=1)
foo.MDR.value_counts()

MDR
False    7239
True     6645
Name: count, dtype: int64

In [55]:
foo

prediction     label      \
DRUG                                                  INH RIF   INH RIF   
id                                   CATALOGUE                            
site.ENA.subj.ERR036186.lab.1.iso.1  CATv1              S   S     S   S   
                                     WHOv1              S   S     S   S   
                                     WHOv2              S   S     S   S   
site.ENA.subj.ERR036187.lab.1.iso.1  CATv1              S   S     S   S   
                                     WHOv1              S   S     S   S   
...                                                   ...  ..   ...  ..   
site.ENA.subj.SRR8651654.lab.1.iso.1 WHOv1              S   R     R   R   
                                     WHOv2              S   R     R   R   
site.ENA.subj.SRR8651662.lab.1.iso.1 CATv1              S   R     R   R   
                                     WHOv1              S   R     R   R   
                                     WHOv2              S   R     R   R   

                                                  MDR  AGREE  
DRUG                                                          
id                                   CATALOGUE                
site.ENA.subj.ERR036186.lab.1.iso.1  CATv1      False  False  
                                     WHOv1      False  False  
                                     WHOv2      False  False  
site.ENA.subj.ERR036187.lab.1.iso.1  CATv1      False  False  
                                     WHOv1      False  False  
...                                               ...    ...  
site.ENA.subj.SRR8651654.lab.1.iso.1 WHOv1       True  False  
                                     WHOv2       True  False  
site.ENA.subj.SRR8651662.lab.1.iso.1 CATv1       True  False  
                                     WHOv1       True  False  
                                     WHOv2       True  False  

[13884 rows x 6 columns]

In [56]:
foo.reset_index(inplace=True)

In [59]:
baa = foo[foo.CATALOGUE=='WHOv1']
baa[baa.MDR].AGREE.value_counts()

AGREE
False    2215
Name: count, dtype: int64

In [61]:
baa.AGREE.value_counts()

AGREE
False    4628
Name: count, dtype: int64